# Cryptocurrency Forecast — Exploratory Data Analysis

This notebook performs exploratory analysis of the cryptocurrency forecasting results.

## Objectives

1. Dataset overview
2. Data quality checks
3. Train/test distribution
4. Historical price behavior
5. Actual vs predicted prices
6. Forecast error analysis
7. Residual analysis
8. Control-limit analysis
9. Model performance comparison

Residuals are defined as:

**Residual = Actual Price - Predicted Price**

Therefore:

- Positive residual → model underpredicted
- Negative residual → model overpredicted


## 1. Configuration


In [0]:
FORECAST_TABLE = "dbx_joshdevph_dev.models.cg_coin_forecasts"
MODEL_PERFORMANCE_TABLE = "dbx_joshdevph_dev.models.cg_coin_model_performance"


## 2. Imports


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


## 3. Load Data


In [0]:
forecast_df = spark.table(FORECAST_TABLE)
performance_df = spark.table(MODEL_PERFORMANCE_TABLE)

print("Forecast table:", FORECAST_TABLE)
print("Performance table:", MODEL_PERFORMANCE_TABLE)


## 4. Dataset Overview


In [0]:
forecast_df.printSchema()


In [0]:
display(
    forecast_df.orderBy(
        "coin_id",
        "timestamp"
    ).limit(5)
)


In [0]:
print(f"Total forecast observations: {forecast_df.count():,}")


### Coins Available


In [0]:
display(
    forecast_df
    .select("coin_id", "vs_currency")
    .distinct()
    .orderBy("coin_id", "vs_currency")
)


### Date Range by Coin


In [0]:
date_range_df = (
    forecast_df
    .groupBy("coin_id", "vs_currency")
    .agg(
        F.min("timestamp").alias("start_timestamp"),
        F.max("timestamp").alias("end_timestamp"),
        F.count("*").alias("observations"),
    )
)

display(date_range_df)


## 5. Data Quality


In [0]:
columns_to_check = [
    "coin_id",
    "timestamp",
    "vs_currency",
    "price",
    "prediction",
    "residual",
]

null_summary_df = forecast_df.select(
    [
        F.sum(
            F.col(column).isNull().cast("int")
        ).alias(column)
        for column in columns_to_check
    ]
)

display(null_summary_df)


### Duplicate Check

Each `coin_id + vs_currency + timestamp` combination should normally represent one observation.


In [0]:
duplicate_df = (
    forecast_df
    .groupBy(
        "coin_id",
        "vs_currency",
        "timestamp"
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_df.count()

print(f"Duplicate timestamps: {duplicate_count:,}")

if duplicate_count > 0:
    display(duplicate_df)


## 6. Train/Test Distribution


In [0]:
train_test_summary_df = (
    forecast_df
    .groupBy(
        "coin_id",
        "dataset_type"
    )
    .agg(
        F.count("*").alias("observations"),
        F.min("timestamp").alias("start_timestamp"),
        F.max("timestamp").alias("end_timestamp"),
    )
    .orderBy(
        "coin_id",
        "dataset_type"
    )
)

display(train_test_summary_df)


### Verify Chronological Split

The maximum training timestamp should occur before the minimum test timestamp for each coin.


In [0]:
split_validation_df = (
    forecast_df
    .groupBy("coin_id")
    .agg(
        F.max(
            F.when(
                F.col("dataset_type") == "train",
                F.col("timestamp")
            )
        ).alias("last_train_timestamp"),

        F.min(
            F.when(
                F.col("dataset_type") == "test",
                F.col("timestamp")
            )
        ).alias("first_test_timestamp"),
    )
    .withColumn(
        "valid_chronological_split",
        F.col("last_train_timestamp") < F.col("first_test_timestamp")
    )
)

display(split_validation_df)


## 7. Price Descriptive Statistics


In [0]:
price_stats_df = (
    forecast_df
    .groupBy(
        "coin_id",
        "dataset_type"
    )
    .agg(
        F.count("*").alias("n"),
        F.avg("price").alias("mean_price"),
        F.stddev_samp("price").alias("std_price"),
        F.min("price").alias("min_price"),
        F.expr("percentile_approx(price, 0.25)").alias("q1"),
        F.expr("percentile_approx(price, 0.5)").alias("median"),
        F.expr("percentile_approx(price, 0.75)").alias("q3"),
        F.max("price").alias("max_price"),
    )
)

display(
    price_stats_df.orderBy(
        "coin_id",
        "dataset_type"
    )
)


## 8. Historical Price Trend

Use Databricks visualization options on the result below:

- Visualization: **Line**
- X-axis: `timestamp`
- Y-axis: `price`
- Series: `coin_id`


In [0]:
display(
    forecast_df
    .select(
        "timestamp",
        "coin_id",
        "price"
    )
    .orderBy(
        "timestamp",
        "coin_id"
    )
    .limit(5)
)


## 9. Actual vs Predicted Prices


In [0]:
coins = [
    row["coin_id"]
    for row in (
        forecast_df
        .select("coin_id")
        .distinct()
        .orderBy("coin_id")
        .collect()
    )
]

print(f"Coins: {coins}")


In [0]:
for coin_id in coins:
    print(f"\n{coin_id.upper()}")

    display(
        forecast_df
        .filter(F.col("coin_id") == coin_id)
        .select(
            "timestamp",
            "dataset_type",
            "price",
            "prediction"
        )
        .orderBy("timestamp")
        .limit(5)
    )


## 10. Residual Descriptive Statistics

A well-behaved forecasting model should generally have residuals centered near zero.


In [0]:
residual_stats_df = (
    forecast_df
    .groupBy(
        "coin_id",
        "dataset_type"
    )
    .agg(
        F.count("*").alias("n"),
        F.avg("residual").alias("mean_residual"),
        F.stddev_samp("residual").alias("std_residual"),
        F.min("residual").alias("min_residual"),
        F.expr("percentile_approx(residual, 0.25)").alias("q1"),
        F.expr("percentile_approx(residual, 0.5)").alias("median"),
        F.expr("percentile_approx(residual, 0.75)").alias("q3"),
        F.max("residual").alias("max_residual"),
    )
)

display(
    residual_stats_df.orderBy(
        "coin_id",
        "dataset_type"
    )
    .limit(5)
)


## 11. Residuals Over Time

Look for:

- Residuals consistently above or below zero
- Increasing residual variability
- Long runs on one side of the center line
- Sudden spikes
- Changes after the train/test boundary


In [0]:
for coin_id in coins:
    print(f"\nResiduals: {coin_id.upper()}")

    display(
        forecast_df
        .filter(F.col("coin_id") == coin_id)
        .select(
            "timestamp",
            "dataset_type",
            "residual"
        )
        .orderBy("timestamp")
        .limit(5)
    )


## 12. Residual Control Chart Data

Control limits were estimated from the **training residuals only**.

The stored limits include:

- Center Line — mean training residual
- ±1 standard deviation
- ±2 standard deviations
- UCL/LCL — ±3 standard deviations

These training limits are then applied to the test residuals to monitor forecast behavior on future observations.


In [0]:
for coin_id in coins:
    print(f"\nControl Chart: {coin_id.upper()}")

    display(
        forecast_df
        .filter(F.col("coin_id") == coin_id)
        .select(
            "timestamp",
            "dataset_type",
            "residual",
            "residual_mean",
            "lower_1_sigma",
            "upper_1_sigma",
            "lower_2_sigma",
            "upper_2_sigma",
            "lcl",
            "ucl"
        )
        .orderBy("timestamp")
        .limit(5)
    )


## 13. Residuals Outside Control Limits


In [0]:
control_status_df = (
    forecast_df
    .withColumn(
        "beyond_control_limit",
        (F.col("residual") > F.col("ucl"))
        | (F.col("residual") < F.col("lcl"))
    )
)

out_of_control_df = (
    control_status_df
    .filter(F.col("beyond_control_limit"))
    .select(
        "coin_id",
        "timestamp",
        "dataset_type",
        "price",
        "prediction",
        "residual",
        "lcl",
        "ucl"
    )
    .orderBy(
        "coin_id",
        "timestamp"
    )
    .limit(5)
)

display(out_of_control_df)


### Number of Out-of-Control Observations


In [0]:
out_of_control_summary_df = (
    control_status_df
    .groupBy(
        "coin_id",
        "dataset_type"
    )
    .agg(
        F.count("*").alias("observations"),
        F.sum(
            F.col("beyond_control_limit").cast("int")
        ).alias("beyond_control_limits"),
    )
    .withColumn(
        "percent_beyond_limits",
        (
            F.col("beyond_control_limits")
            / F.col("observations")
        ) * 100
    )
)

display(
    out_of_control_summary_df.orderBy(
        "coin_id",
        "dataset_type"
    )
    .limit(5)
)


## 14. Forecast Error Metrics

These metrics are calculated directly from the stored forecast observations.

MAE and RMSE remain in the original price/currency units. MAPE is expressed as a percentage.


In [0]:
forecast_metrics_df = (
    forecast_df
    .groupBy(
        "coin_id",
        "dataset_type"
    )
    .agg(
        F.avg(
            F.abs(F.col("residual"))
        ).alias("mae"),

        F.sqrt(
            F.avg(
                F.pow(
                    F.col("residual"),
                    2
                )
            )
        ).alias("rmse"),

        F.avg(
            F.col("residual")
        ).alias("mean_error"),

        (
            F.avg(
                F.when(
                    F.col("price") != 0,
                    F.abs(
                        F.col("residual")
                        / F.col("price")
                    )
                )
            ) * 100
        ).alias("mape"),
    )
)

display(
    forecast_metrics_df.orderBy(
        "coin_id",
        "dataset_type"
    )
    .limit(5)
)


## 15. Compare Train vs Test Error


In [0]:
error_comparison_df = (
    forecast_metrics_df
    .groupBy("coin_id")
    .pivot(
        "dataset_type",
        ["train", "test"]
    )
    .agg(
        F.first("mae").alias("mae"),
        F.first("rmse").alias("rmse"),
        F.first("mean_error").alias("mean_error"),
        F.first("mape").alias("mape"),
    )
)

display(error_comparison_df.limit(5))


## 16. Model Performance Summary


In [0]:
display(
    performance_df.orderBy("coin_id")
)


## 17. Largest Forecast Errors


In [0]:
largest_errors_df = (
    forecast_df
    .withColumn(
        "absolute_residual",
        F.abs("residual")
    )
    .select(
        "coin_id",
        "timestamp",
        "dataset_type",
        "price",
        "prediction",
        "residual",
        "absolute_residual"
    )
    .orderBy(
        F.desc("absolute_residual")
    )
)

display(largest_errors_df.limit(5))


## 18. Largest Test Forecast Errors


In [0]:
display(
    largest_errors_df
    .filter(
        F.col("dataset_type") == "test"
    )
    .orderBy(
        F.desc("absolute_residual")
    )
)


## 19. Residual Change Between Train and Test

Important signals include:

- Test residual mean moving substantially away from zero
- Test residual standard deviation becoming much larger
- Increased RMSE/MAE
- More observations crossing training control limits

These may indicate that the relationship learned during training is becoming less representative of current market behavior.


In [0]:
residual_comparison_df = (
    residual_stats_df
    .groupBy("coin_id")
    .pivot(
        "dataset_type",
        ["train", "test"]
    )
    .agg(
        F.first("mean_residual").alias("mean_residual"),
        F.first("std_residual").alias("std_residual"),
    )
)

display(residual_comparison_df.limit(5))


## 20. EDA Summary

The exploratory analysis focuses on three major areas:

### Price Behavior

- Historical price levels and variability
- Differences between training and test periods
- Actual versus predicted price movements

### Forecast Accuracy

- MAE
- RMSE
- MAPE
- Mean forecast error
- Largest forecast errors

### Forecast Monitoring

- Residual mean
- Residual standard deviation
- ±1σ and ±2σ regions
- ±3σ control limits
- Out-of-control residuals
- Changes in residual behavior between train and test periods

The training residual distribution serves as the baseline for monitoring future forecast performance. Test observations can then be evaluated against the training control limits to identify potential changes in model behavior or the underlying price process.
